# 05. Tomographic focusing by beamforming

Tomographic focusing recovers the reflectivity profile along height from the complex stack. In the production pipeline this is performed by PyRat (`tomo.fusartomo`), which is not available here; this notebook demonstrates the mathematical core of that focusing on a synthetic stack: build steering vectors from kz and a height grid, estimate the sample covariance, and apply the Bartlett (conventional) beamformer to reconstruct the height profile.

**Modules exercised:** pipelines.processing_pipeline.tomogram (beamforming stage performed by tomo.fusartomo), configuration.processing_config (TomogramConfiguration.beamforming_method, height_range)

In [ ]:
import sys
from pathlib import Path

repo_root = Path("../..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except ImportError:
    torch = None

RNG = np.random.default_rng(0)

%matplotlib inline
plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.grid"      : False,
    "image.cmap"     : "viridis",
})

print("repo root:", repo_root)


## Geometry: steering vectors

For a height `h` the stack response is the steering vector `a(h)[t] = exp(1j * kz[t] * h)`. Focusing scans candidate heights and measures how well each steering vector matches the observed signal statistics.

In [ ]:
n_tracks = 8
kz       = np.linspace(0.0, 0.40, n_tracks)

height_range = (-20.0, 80.0)
height_grid  = np.linspace(height_range[0], height_range[1], 256)

steering = np.exp(1.0j * kz[:, None] * height_grid[None, :])
print('steering matrix:', steering.shape, '(tracks x heights)')

## A pixel with two scatterers at different heights

We synthesise many looks of a single pixel containing two scatterers at known heights, with independent speckle per look, so a sample covariance can be formed.

In [ ]:
true_heights = [10.0, 50.0]
true_powers  = [1.0, 0.7]
n_looks      = 200

signal = np.zeros((n_tracks, n_looks), dtype=np.complex64)
for h, p in zip(true_heights, true_powers):
    a      = np.exp(1.0j * kz * h)[:, None]
    amp    = np.sqrt(p) * (RNG.standard_normal(n_looks) + 1.0j * RNG.standard_normal(n_looks)) / np.sqrt(2.0)
    signal += a * amp[None, :]
signal += 0.05 * (RNG.standard_normal(signal.shape) + 1.0j * RNG.standard_normal(signal.shape))
print('signal (tracks x looks):', signal.shape)

## Sample covariance and Bartlett beamformer

The sample covariance averages the outer product over looks (this is exactly what the Boxcar filter approximates spatially). The Bartlett spectrum at height `h` is `a(h)^H R a(h)`.

In [ ]:
R = (signal @ signal.conj().T) / signal.shape[1]
print('covariance R:', R.shape)

bartlett = np.einsum('th,ts,sh->h', steering.conj(), R, steering).real
bartlett = bartlett / bartlett.max()
peak_heights = height_grid[np.argsort(bartlett)[-2:]]
print('two strongest Bartlett peaks at heights:', np.round(np.sort(peak_heights), 1))

## Reconstructed profile

The Bartlett spectrum over the height grid should peak near the two planted heights.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(height_grid, bartlett, label='Bartlett spectrum')
for h in true_heights:
    ax.axvline(h, color='r', ls='--', lw=0.9)
ax.set_xlabel('height (m)'); ax.set_ylabel('normalised power')
ax.set_title('Tomographic focusing along height')
ax.legend()
fig.tight_layout()
plt.show()

## Expected visual outcome

The Bartlett spectrum should show two clear lobes located close to the planted heights of 10 m and 50 m (red dashed lines), demonstrating that beamforming over the kz-defined steering vectors recovers the vertical reflectivity profile.